## Phase 0.2 – Effect of the train/test split on the late delivery model

The same Random Forest classifier is trained and evaluated with three different splits:

1. **Random rows (original approach):** 80/20 stratified random split of order items.
2. **Grouped by Order Id:** all items of an order are placed either in the training or in the test set.
3. **Time-based:** the model is trained on the oldest 80% of orders and tested on the most recent 20%.

Items belonging to the same order almost always share the same label. With a random row split, items of one order end up in both sets, so the model can effectively "recognise" the order instead of learning a generalisable pattern (data leakage). The time-based split reflects real use: the model learns from the past and predicts the future. Therefore, the time-based result is reported as the realistic performance estimate.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

from src.data import load_data
from src.features import DELAY_FEATURES, DELAY_TARGET, delay_preprocessor
from src.evaluation import time_split, classification_metrics

In [2]:
data = load_data()
print(data.shape)

(180519, 51)


In [3]:
X = data[DELAY_FEATURES]
y = data[DELAY_TARGET]
groups = data["Order Id"]
print("Raw feature columns:", X.shape[1])

Raw feature columns: 12


In [4]:
def make_model():
    return Pipeline([
        ("prep", delay_preprocessor()),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ])

def evaluate(train_idx, test_idx, name):
    model = make_model().fit(X.iloc[train_idx], y.iloc[train_idx])
    proba = model.predict_proba(X.iloc[test_idx])[:, 1]
    return {"split": name, **classification_metrics(y.iloc[test_idx], proba)}

all_idx = np.arange(len(data))
results = []

tr, te = train_test_split(all_idx, test_size=0.2, random_state=42, stratify=y)
results.append(evaluate(tr, te, "1) random rows (old)"))

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, y, groups))
results.append(evaluate(tr, te, "2) grouped by Order Id"))

tr, te = time_split(data)
results.append(evaluate(tr, te, "3) time-based (last 20%)"))

print(pd.DataFrame(results).round(3).to_string(index=False))

                   split  accuracy  precision_late  recall_late  f1_late  roc_auc  pr_auc
    1) random rows (old)     0.808           0.849        0.790    0.818    0.899   0.924
  2) grouped by Order Id     0.681           0.730        0.663    0.695    0.750   0.818
3) time-based (last 20%)     0.696           0.756        0.656    0.702    0.762   0.825


In [5]:
# Train on the time-based training set, then score ONE raw order,
# exactly the way the API will receive it.
model = make_model().fit(X.iloc[tr], y.iloc[tr])

new_order = pd.DataFrame([{
    "Customer Zipcode": 725, "order_year": 2017, "order_month": 6,
    "order_week_day": 2, "order_hour": 14,
    "Shipping Mode": "Standard Class", "Market": "Europe",
    "Order Region": "Western Europe", "Category Name": "Cleats",
    "Department Name": "Recreation",   # <- does not exist in DataCo!
    "Customer Segment": "Consumer", "Type": "DEBIT",
}])
print("Late delivery probability:", round(model.predict_proba(new_order)[0, 1], 3))

Late delivery probability: 0.487
